In [ ]:
from RAG import RAG
from langchain_openai import ChatOpenAI
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
import os
import datasets

# Load test dataset
test = datasets.load_dataset('m-ric/huggingface_doc_qa_eval', split="train")
# Your test queries
sample_queries = list(test['question'])
# Your test answers 
expected_responses = list(test['answer']) 

# Initialize your RAG pipeline once
rag = RAG()

# Collect evaluation data
dataset = []

for query,reference in zip(sample_queries,expected_responses):

    relevant_docs = rag.get_most_relevant_docs(query)
    response = rag.generate_answer(query, relevant_docs)
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
    )
# Load into RAGAS
evaluation_dataset = EvaluationDataset.from_list(dataset)

# Configure evaluator LLM pointing at EPFL
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model="swiss-ai/Apertus-70B-Instruct-2509",
        base_url="https://inference.rcp.epfl.ch/v1",
        api_key=os.environ["OPENAI_API_KEY"],
    )
)

# Run evaluation
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness()],
    llm=evaluator_llm,
)
print(result)

/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_53115/2192966059.py:5: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_53115/2192966059.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_53115/2192966059.py:5: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instea

Loading vector database from 'hugging_face_documentation'...
Retrieving top 10 documents for query: 'What architecture is the `tokenizers-linux-x64-musl` binary designed for?
'
Reranking 10 documents, keeping top 5...
--------------------------------------------------
completion_tokens: 31
prompt_tokens:     396
total_tokens:      427
Retrieving top 10 documents for query: 'What is the purpose of the BLIP-Diffusion model?
'
Reranking 10 documents, keeping top 5...
--------------------------------------------------
completion_tokens: 32
prompt_tokens:     2216
total_tokens:      2248
Retrieving top 10 documents for query: 'How can a user claim authorship of a paper on the Hugging Face Hub?
'
Reranking 10 documents, keeping top 5...
--------------------------------------------------
completion_tokens: 85
prompt_tokens:     2519
total_tokens:      2604
Retrieving top 10 documents for query: 'What is the purpose of the /healthcheck endpoint in the Datasets server API?
'
Reranking 10 docume

/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_53115/2192966059.py:38: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


Evaluating:   0%|          | 0/195 [00:00<?, ?it/s]

Exception raised in Job[15]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[73]: TimeoutError()
Exception raised in Job[87]: TimeoutError()
Exception raised in Job[123]: TimeoutError()
Exception raised in Job[129]: TimeoutError()
Exception raised in Job[132]: TimeoutError()
Exception raised in Job[192]: TimeoutError()


{'context_recall': 0.9138, 'faithfulness': 0.9539, 'factual_correctness(mode=f1)': 0.5311}


In [16]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,What architecture is the `tokenizers-linux-x64...,[huggingface/tokenizers/blob/main/bindings/nod...,The `tokenizers-linux-x64-musl` binary is desi...,x86_64-unknown-linux-musl,1.0,1.0,0.67
1,What is the purpose of the BLIP-Diffusion mode...,[huggingface/diffusers/blob/main/docs/source/e...,The purpose of the BLIP-Diffusion model is to ...,The BLIP-Diffusion model is designed for contr...,1.0,1.0,0.40
2,How can a user claim authorship of a paper on ...,[huggingface/hub-docs/blob/main/docs/hub/paper...,To claim authorship of a paper on the Hugging ...,By clicking their name on the corresponding Pa...,1.0,1.0,0.80
3,What is the purpose of the /healthcheck endpoi...,[huggingface/datasets-server/blob/main/service...,The purpose of the /healthcheck endpoint in th...,Ensure the app is running,1.0,1.0,0.00
4,What is the default context window size for Lo...,[huggingface/transformers/blob/main/docs/sourc...,The default context window size for Local Atte...,127 tokens,1.0,1.0,1.00
...,...,...,...,...,...,...,...
60,What is the maximum size of a model checkpoint...,[huggingface/transformers/blob/main/docs/sourc...,"In Transformers version 4.18.0, the maximum si...",10GB,1.0,1.0,1.00
61,What is the purpose of Weights and Biases (W&B...,[gradio-app/gradio/blob/main/guides/cn/04_inte...,The purpose of Weights and Biases (W&B) is to ...,To track their machine learning experiments at...,1.0,1.0,0.50
62,What is the name of the open-source library cr...,[huggingface/blog/blob/main/optimum-onnxruntim...,The name of the open-source library created by...,Optimum,1.0,1.0,0.00
63,What parameter is used to ensure that elements...,[gradio-app/gradio/blob/main/guides/03_buildin...,The parameter used to ensure that elements in ...,equal_height,1.0,1.0,0.50
